# Full-Period TGFSR Search (w=32, r=5, k=160)

Search for TGFSR generators with primitive characteristic polynomial
using `PrimitiveSearch`.

The state size is $k = w \times r = 32 \times 5 = 160$, giving a
period of $2^{160} - 1$.

The search randomizes `m` (feedback offset, $1 \le m < r$) and
`a` (twist coefficient, random 32-bit value), keeping `w` and `r`
fixed.

**Note:** For TGFSR, choosing `r` prime or odd generally yields more
primitive polynomials. Even values of `r` (especially powers of 2)
tend to produce reducible characteristic polynomials.

In [ ]:
from regpoly.search_primitive import PrimitiveSearch
from regpoly.generateur import Generateur

## 1. Parameter specifications

Check which parameters are structural (define $k$) and which are
randomizable.

In [ ]:
specs = Generateur.parameters("TGFSRGen")
print(f"{'Name':<6} {'Type':<8} {'Structural':<12} {'Rand type':<15} {'Rand args'}")
print("-" * 60)
for s in specs:
    print(f"{s['name']:<6} {s['type']:<8} {str(s['structural']):<12} "
          f"{s['rand_type'] or '-':<15} {s['rand_args'] or '-'}")

## 2. Configure and run the search

Create a `PrimitiveSearch` programmatically and run it.
Found generators are saved to `yaml/generators/TGFSRGen/r5_w32.yaml`.

In [ ]:
search = PrimitiveSearch(
    family="TGFSRGen",
    L=32,
    structural_params={"w": 32, "r": 5},
    fixed_params={},           # m and a both randomized
    max_tries=10000,
    max_seconds=None,
    generators_dir="../../yaml/generators",
)

print(f"Output file: {search.output_file}")
results = search.run()

## 3. Inspect results

Load the saved generator file and verify the first entry.

In [ ]:
import yaml

with open(search.output_file) as f:
    data = yaml.safe_load(f)

print(f"Family: {data['family']}")
print(f"Common (structural): {data['common']}")
print(f"Generators found: {len(data['generators'])}")
print()

# Verify the first one
if data["generators"]:
    entry = data["generators"][0]
    gen = Generateur.create("TGFSRGen", 32, **data["common"], **entry)
    print(f"First generator: m={entry['m']}, a={entry['a']}")
    print(f"  k={gen.k}, full period: {gen.is_full_period()}")
    cp = gen.char_poly()
    hw = bin(cp._val).count('1') + 1
    print(f"  Char poly Hamming weight: {hw}")

## 4. Summary table

In [ ]:
k = data["common"]["w"] * data["common"]["r"]
print(f"{'#':<4} {'m':<4} {'a':<12} {'a (hex)'}")
print("-" * 35)
for i, entry in enumerate(data["generators"], 1):
    print(f"{i:<4} {entry['m']:<4} {entry['a']:<12} 0x{entry['a']:08x}")
print(f"\nTotal: {len(data['generators'])} generators with k={k}, period 2^{k}-1")

## 5. Run again with fixed m

Fix `m=2` and only randomize `a`.

In [ ]:
search_m2 = PrimitiveSearch(
    family="TGFSRGen",
    L=32,
    structural_params={"w": 32, "r": 5},
    fixed_params={"m": 2},     # m fixed, a randomized
    max_tries=10000,
    max_seconds=None,
    generators_dir="../../yaml/generators",
)

results_m2 = search_m2.run()

## 6. Equidistribution with MK tempering

For each full-period generator, try 50 random Matsumoto-Kurita
tempering parameter sets (random `b` and `c` masks). Keep and save
those with dimension gaps sum less than 100.

In [ ]:
import random
import yaml
from regpoly.combinaison import Combinaison
from regpoly.transformation import Transformation
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest, METHOD_MATRICIAL
)
from regpoly.tested_generator import save_tested_generator

INT_MAX = 2**31 - 1
SE_THRESHOLD = 100
NBTRIES = 50

# Reload the generators file (includes results from both search runs)
with open(search.output_file) as f:
    data = yaml.safe_load(f)

family = data["family"]
common = data["common"]
L = common["w"]
n_gens = len(data["generators"])

print(f"Testing {n_gens} generators x {NBTRIES} tempering tries "
      f"= {n_gens * NBTRIES} tests")
print(f"Threshold: se < {SE_THRESHOLD}")
print()

good = []

for i, entry in enumerate(data["generators"]):
    gen = Generateur.create(family, L, **common, **entry)

    for t in range(NBTRIES):
        # Random MK tempering: b and c are random L-bit masks
        b = random.getrandbits(L)
        c = random.getrandbits(L)
        temper_params = {"w": L, "eta": 7, "mu": 15, "b": b, "c": c}
        trans = Transformation.create("tempMK", **temper_params)

        comb = Combinaison(J=1, Lmax=L)
        comb.components[0].add_gen(gen)
        comb.components[0].add_trans(trans)
        comb.reset()

        test = EquidistributionTest(L=L, delta=[INT_MAX] * (L + 1),
                                    mse=INT_MAX, method=METHOD_MATRICIAL)
        result = test.run(comb)

        if result.se < SE_THRESHOLD:
            # Save to testedgenerators/
            test_results = {"equidistribution": {"se": result.se}}
            ecart = {l: result.ecart[l]
                     for l in range(1, L + 1) if result.ecart[l] != 0}
            if ecart:
                test_results["equidistribution"]["ecart"] = ecart
            if result.is_me():
                test_results["equidistribution"]["status"] = "ME"

            path = save_tested_generator(
                "../../yaml/testedgenerators", "equidist", comb, test_results)

            good.append({
                "gen": entry,
                "b": b, "c": c,
                "se": result.se,
                "is_me": result.is_me(),
                "path": path,
            })
            print(f"  gen {i+1:>3d} try {t+1:>2d}: "
                  f"m={entry['m']}, a=0x{entry['a']:08x}, "
                  f"b=0x{b:08x}, c=0x{c:08x} -> se={result.se}"
                  f"{'  ME!' if result.is_me() else ''}")

print(f"\nFound {len(good)} good generators (se < {SE_THRESHOLD}) "
      f"out of {n_gens * NBTRIES} tests")

## 7. Results summary

Sorted by dimension gaps sum. Saved files can be loaded by any test.

In [ ]:
if good:
    ranked = sorted(good, key=lambda x: x["se"])
    print(f"{'#':<4} {'m':<4} {'a (hex)':<12} {'b (hex)':<12} {'c (hex)':<12} "
          f"{'se':>4} {'ME':>3}  {'file'}")
    print("-" * 90)
    for rank, g in enumerate(ranked, 1):
        print(f"{rank:<4} {g['gen']['m']:<4} 0x{g['gen']['a']:08x} "
              f"0x{g['b']:08x} 0x{g['c']:08x} "
              f"{g['se']:>4} {'yes' if g['is_me'] else '':>3}  "
              f"{g['path']}")
else:
    print(f"No generators found with se < {SE_THRESHOLD}.")

## 8. Equidistribution table for the best generator

Reload the best saved file and show its full dimension gaps table.

In [ ]:
from regpoly.tested_generator import load_tested_generator

if good:
    best = sorted(good, key=lambda x: x["se"])[0]
    comb, saved_results = load_tested_generator(best["path"])

    print(f"Loaded: {best['path']}")
    print(f"  m={best['gen']['m']}, a=0x{best['gen']['a']:08x}")
    print(f"  tempering: b=0x{best['b']:08x}, c=0x{best['c']:08x}")
    print()

    # Re-run equidistribution to get the full result object
    test = EquidistributionTest(L=comb.L, delta=[INT_MAX] * (comb.L + 1),
                                mse=INT_MAX, method=METHOD_MATRICIAL)
    result = test.run(comb)

    print(f"Dimension gaps sum = {result.se}")
    if result.is_me():
        print("==> ME generator!")
    print()

    table_str, _ = result.display_table(comb, 'l')
    print(table_str)
else:
    print(f"No generators found with se < {SE_THRESHOLD}.")